# AWS PrivateLink & Snowflake — End-to-End Setup

This notebook walks through every step required to configure **AWS PrivateLink** for inbound private connectivity to a Snowflake account on AWS.

```
┌─────────────────────┐         ┌───────────────────────────┐         ┌─────────────────────┐
│   Your AWS VPC      │         │    AWS PrivateLink        │         │   Snowflake VPC     │
│                     │         │                           │         │                     │
│  ┌───────────────┐  │         │  ┌─────────────────────┐  │         │  ┌───────────────┐  │
│  │  Application  │──│────────►│  │  VPC Endpoint       │──│────────►│  │  Snowflake    │  │
│  │  / Client     │  │  ENI    │  │  (Interface VPCE)    │  │  PL     │  │  Service      │  │
│  └───────────────┘  │         │  └─────────────────────┘  │         │  └───────────────┘  │
│                     │         │                           │         │                     │
│  ┌───────────────┐  │         │  ┌─────────────────────┐  │         │                     │
│  │  Route 53     │  │         │  │  S3 Gateway         │  │         │                     │
│  │  Private Zone │  │         │  │  Endpoint (optional)│  │         │                     │
│  └───────────────┘  │         │  └─────────────────────┘  │         │                     │
└─────────────────────┘         └───────────────────────────┘         └─────────────────────┘
```

### Prerequisites

| Requirement | Details |
|---|---|
| **Snowflake Edition** | Business Critical (or higher) |
| **Snowflake Role** | `ACCOUNTADMIN` |
| **AWS Permissions** | VPC Admin — create VPC endpoints, security groups, Route 53 zones |
| **AWS CLI** | Installed and authenticated (`aws configure` or SSO) |
| **AWS IAM** | Permission to call `sts:GetFederationToken` |
| **Warehouse** | Active warehouse for running Snowflake queries |

---
## Part 1 — AWS: Generate Federation Token

Before enabling PrivateLink in Snowflake, you must generate an AWS STS federation token. This token proves you own the AWS account that will create the VPC endpoint.

### 1.1 — Generate the Federation Token

Run the following in your terminal (not in this notebook — it requires the AWS CLI):

```bash
aws sts get-federation-token --name snowflake-privatelink
```

**Why:** Snowflake uses this token to verify that the caller owns the AWS account requesting PrivateLink authorization.

> **Impact:** Without a valid federation token, `SYSTEM$AUTHORIZE_PRIVATELINK` will fail. The token expires after **12 hours**.

> **Doc:** [How to retrieve a Federation Token](https://community.snowflake.com/s/article/How-to-retrieve-a-Federation-Token-from-AWS-for-PrivateLink-Self-Service)

### 1.2 — Extract Key Values from the Token

From the JSON output, note two values:

| Field | Where to find it | Example |
|---|---|---|
| **AWS Account ID** (12-digit) | `FederatedUser.FederatedUserId` — digits before the colon | `185012345678` |
| **Full Token JSON** | Entire JSON output from the command | `{"Credentials": {...}, ...}` |

> **Impact:** Using the wrong AWS Account ID will authorize the wrong account for PrivateLink access.

---
## Part 2 — Snowflake: Enable PrivateLink

These steps run inside Snowflake to authorize your AWS account and retrieve the endpoint configuration.

### Step 1 — Authorize AWS PrivateLink

Enables AWS PrivateLink for your Snowflake account by linking it to your AWS account.

**Why:** This is the master switch. Snowflake must explicitly authorize your AWS account before any VPC endpoint can connect.

> **Impact:** Skipping this step means VPC endpoints will be rejected by Snowflake.

Replace `<AWS_ACCOUNT_ID>` with your 12-digit AWS Account ID and `<FEDERATION_TOKEN_JSON>` with the full JSON output from Step 1.1.

> **Doc:** [SYSTEM$AUTHORIZE_PRIVATELINK](https://docs.snowflake.com/en/sql-reference/functions/system_authorize_privatelink)

In [ ]:
%%sql -r dataframe_1
USE ROLE ACCOUNTADMIN;

SELECT SYSTEM$AUTHORIZE_PRIVATELINK(
    '<AWS_ACCOUNT_ID>',
    '<FEDERATION_TOKEN_JSON>'
);

### Step 2 — Verify PrivateLink Authorization

Confirms PrivateLink was successfully authorized for your AWS account.

**Why:** Ensures the authorization call in Step 1 succeeded before proceeding.

> **Impact:** If this returns an error, the federation token may have expired or the AWS Account ID was incorrect.

> **Doc:** [SYSTEM$GET_PRIVATELINK](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink)

In [ ]:
%%sql -r dataframe_2
SELECT SYSTEM$GET_PRIVATELINK(
    '<AWS_ACCOUNT_ID>',
    '<FEDERATION_TOKEN_JSON>'
);

### Step 3 — Retrieve PrivateLink Configuration

Returns all endpoint identifiers, URLs, and service IDs needed for the AWS VPC endpoint and DNS records.

**Why:** This single function returns every value you need for the AWS side — the VPCE service ID, account URLs, OCSP endpoints, and Snowsight URLs.

> **Impact:** You cannot create the VPC endpoint without the `privatelink-vpce-id` value returned here.

> **Doc:** [SYSTEM$GET_PRIVATELINK_CONFIG](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_config)

In [ ]:
%%sql -r dataframe_3
SELECT SYSTEM$GET_PRIVATELINK_CONFIG();

### Step 4 — Flatten Configuration for Easy Reading

Parses the JSON into a readable key-value table.

**Why:** The raw JSON is hard to read. Flattening gives you a clean table to copy values for the AWS console.

**Key values to copy:**

| Key | Used For |
|---|---|
| `privatelink-vpce-id` | Creating the AWS VPC endpoint |
| `privatelink-account-url` | DNS CNAME for Snowflake account access |
| `privatelink-ocsp-url` | DNS CNAME for OCSP cache server |
| `snowsight-privatelink-url` | DNS CNAME for Snowsight UI |
| `regionless-snowsight-privatelink-url` | DNS CNAME for org-level Snowsight |
| `regionless-privatelink-account-url` | DNS CNAME for org-level account access |

In [ ]:
%%sql -r dataframe_4
SELECT key, value
FROM TABLE(FLATTEN(INPUT => PARSE_JSON(SYSTEM$GET_PRIVATELINK_CONFIG())));

---
## Part 3 — AWS: Create VPC Endpoint

These steps are performed in the **AWS Console** or via the **AWS CLI**.

### 3.1 — Create VPC Interface Endpoint

Using the `privatelink-vpce-id` value from Step 4, create a VPC interface endpoint.

**AWS Console:** VPC → Endpoints → Create Endpoint

| Setting | Value |
|---|---|
| **Service category** | Other endpoint services |
| **Service name** | Paste `privatelink-vpce-id` value |
| **VPC** | Select the VPC where your applications run |
| **Subnets** | Select subnets in at least 2 availability zones |
| **Security group** | Select or create a security group (see 3.2) |

```bash
aws ec2 create-vpc-endpoint \
    --vpc-id vpc-0abc123def456 \
    --service-name <privatelink-vpce-id> \
    --vpc-endpoint-type Interface \
    --subnet-ids subnet-aaa111 subnet-bbb222 \
    --security-group-ids sg-0xyz789
```

**Why:** The VPC interface endpoint creates an ENI in your subnets that routes traffic privately to Snowflake.

> **Impact:** Without this endpoint, all traffic to Snowflake traverses the public internet.

> **Doc:** [AWS VPC Endpoints](https://docs.aws.amazon.com/vpc/latest/userguide/vpc-endpoints.html)

### 3.2 — Configure Security Group

The security group must allow inbound traffic on ports **443** (HTTPS) and **80** (OCSP cache).

| Rule | Port | Source |
|---|---|---|
| HTTPS | 443 | Your VPC CIDR (e.g. `10.0.0.0/16`) |
| OCSP Cache | 80 | Your VPC CIDR (e.g. `10.0.0.0/16`) |

```bash
aws ec2 authorize-security-group-ingress --group-id sg-0xyz789 --protocol tcp --port 443 --cidr 10.0.0.0/16
aws ec2 authorize-security-group-ingress --group-id sg-0xyz789 --protocol tcp --port 80 --cidr 10.0.0.0/16
```

**Why:** Port 443 handles all Snowflake traffic. Port 80 is required for the OCSP certificate cache server.

> **Impact:** Forgetting port 80 causes intermittent TLS/OCSP errors in Snowflake clients.

### 3.3 — Create S3 Gateway Endpoint (Optional)

If your VPC blocks public internet access, Snowflake clients need a path to S3 for internal stage operations.

```bash
aws ec2 create-vpc-endpoint \
    --vpc-id vpc-0abc123def456 \
    --service-name com.amazonaws.<REGION>.s3 \
    --vpc-endpoint-type Gateway \
    --route-table-ids rtb-0abc123
```

**Why:** Snowflake clients communicate with S3 for data loading/unloading and query result caching.

> **Impact:** Without S3 access, `PUT`, `GET`, and large query results will fail with timeout errors.

> **Doc:** [Gateway endpoints for Amazon S3](https://docs.aws.amazon.com/vpc/latest/userguide/vpc-endpoints-s3.html)

---
## Part 4 — AWS: Configure DNS

Snowflake clients must resolve Snowflake hostnames to the **private IP addresses** of your VPC endpoint.

### 4.1 — Create Route 53 Private Hosted Zone

| Setting | Value |
|---|---|
| **Domain name** | `privatelink.snowflakecomputing.com` |
| **Type** | Private hosted zone |
| **VPC** | Select the same VPC as your endpoint |

```bash
aws route53 create-hosted-zone \
    --name privatelink.snowflakecomputing.com \
    --vpc VPCRegion=<REGION>,VPCId=vpc-0abc123def456 \
    --caller-reference $(date +%s) \
    --hosted-zone-config PrivateZone=true
```

**Why:** Without a private hosted zone, DNS resolves to public IPs, bypassing PrivateLink entirely.

> **Impact:** Traffic goes over the public internet instead of PrivateLink.

> **Doc:** [Route 53 Private Hosted Zones](https://docs.aws.amazon.com/Route53/latest/DeveloperGuide/hosted-zones-private.html)

### 4.2 — Create CNAME Records

Create CNAME records pointing each Snowflake hostname to the **VPC endpoint DNS name**.

| CNAME Record Name | Source Key from Step 4 |
|---|---|
| `<acct>.<region>.privatelink.snowflakecomputing.com` | `privatelink-account-url` |
| `ocsp.<acct>.<region>.privatelink.snowflakecomputing.com` | `privatelink-ocsp-url` |
| `app.<region>.privatelink.snowflakecomputing.com` | `snowsight-privatelink-url` |
| `app.privatelink.snowflakecomputing.com` | `regionless-snowsight-privatelink-url` |
| `<org>-<acct>.privatelink.snowflakecomputing.com` | `regionless-privatelink-account-url` |
| `ocsp.<org>-<acct>.privatelink.snowflakecomputing.com` | `regionless-privatelink-ocsp-url` |

**Why:** CNAME records redirect DNS lookups to your VPC endpoint's private IPs.

> **Impact:** Missing CNAMEs mean specific features (Snowsight, OCSP) won't work over PrivateLink.

> **Doc:** [SYSTEM$GET_PRIVATELINK_CONFIG](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_config)

---
## Part 5 — Snowflake: Verification & Testing

Run these queries to confirm the PrivateLink setup is working correctly.

### Step 5 — Check PrivateLink Endpoint Status

**Why:** After creating the VPC endpoint in AWS, there can be a propagation delay. This shows whether Snowflake sees the endpoint as `available`.

> **Impact:** Status `pendingAcceptance` or `rejected` means the VPC endpoint or authorization is misconfigured.

> **Doc:** [SYSTEM$GET_PRIVATELINK_ENDPOINTS_INFO](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_endpoints_info)

In [ ]:
%%sql -r dataframe_5
SELECT SYSTEM$GET_PRIVATELINK_ENDPOINTS_INFO();

### Step 6 — Get PrivateLink Allowlist

Returns all hostnames and ports that clients need to reach over PrivateLink.

**Why:** Validate that DNS CNAME records and security group rules cover all required hostnames.

> **Impact:** Missing entries may cause intermittent connectivity failures.

> **Doc:** [SYSTEM$ALLOWLIST_PRIVATELINK](https://docs.snowflake.com/en/sql-reference/functions/system_allowlist_privatelink)

In [ ]:
%%sql -r dataframe_6
SELECT value:type::VARCHAR AS type,
       value:host::VARCHAR AS host,
       value:port::INTEGER AS port
FROM TABLE(FLATTEN(INPUT => PARSE_JSON(SYSTEM$ALLOWLIST_PRIVATELINK())));

### Step 7 — Identify Account Region

**Why:** VPC endpoint and DNS records are region-specific. Confirms your Snowflake region.

> **Impact:** Mismatched regions require cross-region PrivateLink (different steps and pricing).

In [ ]:
%%sql -r dataframe_7
SELECT CURRENT_ACCOUNT() AS account,
       CURRENT_REGION()  AS snowflake_region,
       CURRENT_ROLE()    AS current_role;

### Step 8 — Map Snowflake Region to AWS Region

**Why:** Snowflake uses its own region naming. You need the AWS-native region name for VPC configuration.

> **Impact:** Wrong region in your VPC endpoint configuration will cause connection failures.

In [ ]:
%%sql -r dataframe_8
SHOW REGIONS;

---
## Part 6 — Snowflake: Harden Access (Recommended)

After verifying PrivateLink works, block public access so all connections **must** come through the VPC endpoint.

### Step 9 — Create Network Policy

**Why:** Without a network policy, users can still connect over the public internet.

> **Impact:** Once activated, **all users** must connect via PrivateLink. Verify PrivateLink works first!

Replace `<YOUR_VPC_CIDR>` with your VPC's CIDR block (e.g. `10.0.0.0/16`).

> **Doc:** [Network Policies](https://docs.snowflake.com/en/user-guide/network-policies)

In [ ]:
%%sql -r dataframe_9
CREATE OR REPLACE NETWORK POLICY privatelink_only_policy
    ALLOWED_IP_LIST = ('<YOUR_VPC_CIDR>')
    COMMENT = 'Restrict access to PrivateLink VPC endpoints only';

### Step 10 — Activate Network Policy on Account

**Why:** Creating a network policy does not enforce it — you must activate it at the account level.

> **Impact:** This is a **lockout risk**. If PrivateLink is not working when you activate this, you cannot connect. Test first!

> **Doc:** [ALTER ACCOUNT](https://docs.snowflake.com/en/sql-reference/sql/alter-account)

In [ ]:
%%sql -r dataframe_10
ALTER ACCOUNT SET NETWORK_POLICY = privatelink_only_policy;

### Step 11 — Verify Network Policy

**Why:** Quick sanity check that the policy was applied.

> **Impact:** If this shows no policy, the ALTER ACCOUNT command may not have executed.

In [ ]:
%%sql -r dataframe_11
SHOW PARAMETERS LIKE 'NETWORK_POLICY' IN ACCOUNT;

---
## Part 7 — Configure Snowflake Clients

After PrivateLink is established, clients must use the **privatelink** hostname format.

### Client Connection Strings

| Client | Connection Format |
|---|---|
| **SnowSQL / CLI** | `--accountname <acct_id>.privatelink` |
| **JDBC** | `jdbc:snowflake://<acct_id>.<region>.privatelink.snowflakecomputing.com` |
| **ODBC** | `server=<acct_id>.<region>.privatelink.snowflakecomputing.com` |
| **Python** | `account='<acct_id>.<region>.privatelink'` |
| **Spark** | Full hostname: `<acct_id>.<region>.privatelink.snowflakecomputing.com` |

### Minimum Client Versions for OCSP Cache

| Client | Minimum Version |
|---|---|
| Snowflake CLI | 3.0.0 |
| SnowSQL | 1.1.57 |
| Python Connector | 1.8.2 |
| JDBC Driver | 3.8.3 |
| ODBC Driver | 2.19.3 |

> **Doc:** [Configure clients](https://docs.snowflake.com/en/user-guide/admin-security-privatelink#configure-your-snowflake-clients)

---
## Part 8 — Troubleshooting

| Symptom | Likely Cause | Fix |
|---|---|---|
| `SYSTEM$AUTHORIZE_PRIVATELINK` fails | Federation token expired (>12h) | Regenerate with `aws sts get-federation-token` |
| Account mismatch error | Wrong AWS Account ID | Verify 12-digit ID from `FederatedUserId` |
| VPC endpoint stuck in `pending` | Authorization not complete | Re-run `SYSTEM$AUTHORIZE_PRIVATELINK`; verify `privatelink-vpce-id` |
| DNS resolves to public IPs | CNAME records missing | Check Route 53 private hosted zone and CNAME records |
| TLS/OCSP errors | Port 80 blocked or OCSP CNAME missing | Add port 80 to security group; add OCSP CNAME |
| Snowsight not loading | Snowsight CNAMEs missing | Add `snowsight-privatelink-url` and `regionless-snowsight-privatelink-url` |
| Client timeouts | Security group too restrictive | Allow ports 443 + 80 from VPC CIDR |
| Locked out after network policy | Policy applied before PrivateLink verified | Contact Snowflake Support |
| `PUT`/`GET` fail | No S3 endpoint | Create S3 gateway/interface endpoint (Part 3, 3.3) |
| Cross-region failures | Endpoint not cross-region enabled | Enable in AWS console; select Snowflake's region |

---
## Part 9 — Cleanup

All statements are **commented out** to prevent accidental execution.

In [ ]:
%%sql -r dataframe_12
-- ============================================================
-- CLEANUP: Uncomment to tear down PrivateLink configuration
-- ============================================================

-- ALTER ACCOUNT UNSET NETWORK_POLICY;
-- DROP NETWORK POLICY IF EXISTS privatelink_only_policy;

-- SELECT SYSTEM$REVOKE_PRIVATELINK(
--     '<AWS_ACCOUNT_ID>',
--     '<FEDERATION_TOKEN_JSON>'
-- );

-- AWS-side cleanup (run in terminal):
-- aws ec2 delete-vpc-endpoints --vpc-endpoint-ids vpce-0abc123def456
-- aws route53 delete-hosted-zone --id Z0123456789ABCDEF

SELECT 'Cleanup statements are commented out' AS status;

---
## References

| Topic | Link |
|---|---|
| AWS PrivateLink & Snowflake | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/admin-security-privatelink) |
| SYSTEM$AUTHORIZE_PRIVATELINK | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_authorize_privatelink) |
| SYSTEM$GET_PRIVATELINK_CONFIG | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_get_privatelink_config) |
| SYSTEM$ALLOWLIST_PRIVATELINK | [docs.snowflake.com](https://docs.snowflake.com/en/sql-reference/functions/system_allowlist_privatelink) |
| Network Policies | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/network-policies) |
| SnowCD (Connectivity Diagnostic) | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/snowcd) |
| AWS VPC Endpoints | [docs.aws.amazon.com](https://docs.aws.amazon.com/vpc/latest/userguide/vpc-endpoints.html) |
| AWS PrivateLink Pricing | [aws.amazon.com](https://aws.amazon.com/privatelink/pricing) |
| Troubleshooting (Community) | [community.snowflake.com](https://community.snowflake.com/s/article/Troubleshooting-Snowflake-self-service-functions-for-AWS-PrivateLink) |
| FAQ: PrivateLink Self-Service | [community.snowflake.com](https://community.snowflake.com/s/article/PrivateLink-Self-Service-with-AWS) |
| Pinning Private Endpoints | [docs.snowflake.com](https://docs.snowflake.com/en/user-guide/pin-private-endpoints) |